## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 01: Data Collection**

---

**Scope:** Downloads Bank Nifty and Nifty 50 index OHLCV, constructs the settlement-weekday expiry calendar and trading holiday list, and (one-time, documented but not re-run here) pulls the full NSE futures-and-options Bhavcopy archive.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, F&O Bhavcopy archive (NOTE: the rule-based expiry calendar it builds is NOT used for inference)
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical features (true-calendar expiry features) and the true-expiry index panel
3. **03_weekly_options_discontinuation_identification.ipynb** -- true expiry dates per index and a calendar audit
4. **04_primary_diff_in_diff_analysis.ipynb** -- corrected symbol-month DiD (turnover, expiry-day turnover, expiry-day share)
5. **05_robustness_checks.ipynb** -- placebo, window sensitivity, substitution range, participant mix, futures market quality
6. **06_expiry_volume_event_study.ipynb** -- expiry-day concentration for six NSE/BSE indices
7. **07_walk_forward_predictability.ipynb** -- secondary: leakage-free predictability test on the true calendar (null)
8. **08_interrupted_time_series.ipynb** -- secondary: settlement-weekday ITS plus clean weekday-only contrasts (null)
9. **09_figures.ipynb** -- displays the corrected figures
10. **10_option_microstructure_and_placebo.ipynb** -- option-level outcomes, stock-F&O placebo, dose-response

**Current location:** **01_data_collection.ipynb** <- *you are here*


### 1. Index OHLCV, expiry calendar, and holidays

`src.data_loader.load_data_pipeline` downloads daily Bank Nifty OHLCV from Yahoo Finance, programmatically builds the monthly expiry calendar across the Thursday -> Wednesday -> Tuesday settlement-weekday reforms (rolling back for exchange holidays), and derives the holiday list from the trading calendar itself.

In [ ]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

In [ ]:
from src.data_loader import load_data_pipeline

df_ohlcv, df_expiry, df_holidays = load_data_pipeline()
print(f'OHLCV: {len(df_ohlcv)} rows, {df_ohlcv["Date"].min()} to {df_ohlcv["Date"].max()}')
print(f'Expiry calendar: {len(df_expiry)} months')
print(f'Holidays: {len(df_holidays)} days')
df_expiry.tail(10)

### 2. NSE futures-and-options Bhavcopy archive

The primary analysis needs daily per-contract volume, traded value, and open interest for every Bank Nifty and Nifty 50 futures/options contract, which the index OHLCV above does not provide. This comes from NSE's own daily Bhavcopy archive, spanning a legacy CSV format (pre 2024-07-08) and the newer UDiFF format.

Downloading the full 2020-2026 archive (~1,700 trading days) is a one-time, rate-limited step and is **not re-run in this notebook**. To (re)build it from scratch:

```bash
python -m scripts.download_fo_bhavcopy 2020-01-01 2026-10-18
```

The cell below just reports what is already on disk.

In [ ]:
legacy_dir = config.RAW_DATA_DIR / 'fo_bhavcopy' / 'legacy'
udiff_dir = config.RAW_DATA_DIR / 'fo_bhavcopy' / 'udiff'
n_legacy = len(list(legacy_dir.glob('*.zip'))) if legacy_dir.exists() else 0
n_udiff = len(list(udiff_dir.glob('*.zip'))) if udiff_dir.exists() else 0
print(f'Legacy-format Bhavcopy files on disk: {n_legacy}')
print(f'UDiFF-format Bhavcopy files on disk: {n_udiff}')
print(f'Total: {n_legacy + n_udiff}')